# What the Data Can't Support
Some results in `04_production_metrics` and `05_lowest_performing_run` rest on choices the data can't confirm. Each section below says why, what data would settle it, and shows the evidence.

In [1]:
import contextlib
import io

import pandas as pd
from IPython.display import display

# Run 04 (which runs 03 and 02) to get the cleaned tables, the matched readings and stops, and the metrics.
# Its printed output is hidden here.
with contextlib.redirect_stdout(io.StringIO()):
    get_ipython().run_line_magic("run", "04_production_metrics.ipynb")


def show(frame):
    return frame.style.hide(axis="index").format(precision=1, na_rep="")

## 1. Which run performed worst
**Why it isn't reliable:** the answer depends on how the `C010` drop is read. As a glitch (the decision in `03`), `R101` is lowest. Under the other reasonable readings, a different run is.

**What would settle it:** a record of when counters were reset.

Target attainment under each reading:

In [2]:
def cases(run, c010_is_reset=False, use_reading_before_start=False):
    # Cases from the counter, with the same glitch rule as 03 unless an alternative is switched on
    start, end = ts(run["start_time"]), ts(run["end_time"])
    machine = sorted((r for r in counter_readings if r["equipment_id"] == run["equipment_id"] and isinstance(r["count"], int)),
                     key=lambda r: r["recorded_at"])
    inside = [r for r in machine if start <= ts(r["recorded_at"]) <= end]
    before = [r for r in machine if ts(r["recorded_at"]) < start]
    if use_reading_before_start and before and ts(inside[0]["recorded_at"]) > start:
        inside = [before[-1]] + inside
    total, last = 0, inside[0]["count"]
    for r in inside[1:]:
        if r["count"] >= last:
            total, last = total + r["count"] - last, r["count"]
        elif c010_is_reset and r["reading_id"] == "C010":
            total, last = total + r["count"], r["count"]
    return total


scenarios = {
    "These decisions": {},
    "C010 is a reset": {"c010_is_reset": True},
    "C010 is a reset and C001/C014 are used": {"c010_is_reset": True, "use_reading_before_start": True},
}
done = metrics[metrics["run_end"].notna()].set_index("run_id")
completed_runs = [run for run in production_runs if run["end_time"] is not None]
attainment = pd.DataFrame(
    {name: [cases(run, **rules) / done.loc[run["run_id"], "operating_minutes"] * 60
            / run["target_hourly_rate"] * 100 for run in completed_runs]
     for name, rules in scenarios.items()},
    index=pd.Index([run["run_id"] for run in completed_runs], name="run_id"),
)
for name in scenarios:
    print(f"{name}: lowest is {attainment[name].idxmin()} ({attainment[name].min():.1f}%)")
attainment.style.format("{:.1f}%")

These decisions: lowest is R101 (49.5%)
C010 is a reset: lowest is R102 (72.7%)
C010 is a reset and C001/C014 are used: lowest is R103 (73.4%)


,These decisions,C010 is a reset,C010 is a reset and C001/C014 are used
run_id,,,
R100,77.8%,77.8%,94.4%
R101,49.5%,91.1%,91.1%
R102,72.7%,72.7%,96.1%
R103,73.4%,73.4%,73.4%


## 2. `R101`'s cases
**Why it isn't reliable:** `C010` and `C011` are ignored as glitches, so nothing after 14:40 is counted, but all of `R101`'s operating time is. If `C010` were a reset, the cases made between 14:40 and the reset would still be unknown, so even 460 would only be a minimum.

**What would settle it:** a record of counter resets, or more frequent readings.

In [3]:
show(pd.DataFrame([r for r in readings if r["run_id"] == "R101"])[["reading_id", "time", "count", "used", "why_not"]])

reading_id,time,count,used,why_not
C007,14:10,1510,yes,
C008,14:30,1680,yes,
C009,14:40,1760,yes,
C010,14:50,25,no,glitch: count dropped
C011,15:10,210,no,glitch: count dropped


## 3. `R100`'s and `R102`'s cases
**Why they aren't reliable:** the readings taken just before each run starts (`C001` at 12:59, `C014` at 13:14) are left out. Counting starts 10 minutes into `R100` and 15 minutes into `R102`, but the full run time still counts, so their attainment is understated.

**What would settle it:** a counter reading at each run's start.

In [4]:
start_ids = ["C001", "C002", "C014", "C015"]
show(pd.DataFrame([r for r in readings if r["reading_id"] in start_ids])[
    ["reading_id", "equipment_id", "time", "count", "run_id", "used", "why_not"]])

reading_id,equipment_id,time,count,run_id,used,why_not
C001,PKG-01,12:59,1000,,no,outside every run
C002,PKG-01,13:10,1090,R100,yes,
C014,PKG-02,13:14,500,,no,outside every run
C015,PKG-02,13:30,590,R102,yes,


## 4. Any metric for `R104`
**Why it isn't reliable:** `R104` has no end time. The operator stop `D010` at 15:34 doesn't prove the run ended: the count still rose from 210 at 15:20 to 360 at 15:40, and the data doesn't show when.

**What would settle it:** a recorded end time.

In [5]:
display(show(pd.DataFrame([r for r in readings if r["run_id"] == "R104"])[["reading_id", "time", "count", "why_not"]]))
show(pd.DataFrame([e for e in events if e["run_id"] == "R104"])[["event_id", "time", "reason", "note"]])

reading_id,time,count,why_not
C012,15:20,210,run has no end time
C013,15:40,360,run has no end time


event_id,time,reason,note
D010,15:34–no end,Operator stop,run has no end time


## 5. What caused the downtime
**Why it isn't reliable:** half the stops are `Unknown`, and the material reason is recorded three different ways (`Material unavailable`, `material`, `MATL`), so downtime can't be reliably totalled by cause.

**What would settle it:** standard reason codes.

Stops and minutes by reason, as recorded (`D010` has no end, so it adds no minutes):

In [6]:
rows = []
for e in downtime_events:
    rows.append({"reason": e["reason"],
                 "minutes": minutes(ts(e["start_time"]), ts(e["end_time"])) if e["end_time"] else None})
by_reason = pd.DataFrame(rows).groupby("reason", sort=False).agg(stops=("reason", "size"), minutes=("minutes", lambda m: m.sum(min_count=1)))
by_reason.sort_values("stops", ascending=False).style.format({"minutes": "{:.1f}"}, na_rep="")

,stops,minutes
reason,,
Unknown,5,9.6
Material unavailable,1,6.0
material,1,4.0
MATL,1,3.0
Jam,1,5.0
Operator stop,1,


## 6. Why `R103` fell short of target
**Why it isn't reliable:** `R103` logged only 3.6 minutes of downtime, but even per operating hour it ran well below its target. The data can't say whether it ran slowly or had stops nobody recorded.

**What would settle it:** machine speed or state data.

In [7]:
show(metrics[metrics["run_id"] == "R103"][[
    "run_id", "cases_produced", "downtime_minutes", "operating_minutes", "operating_throughput_cph",
    "target_rate_cases_per_hour", "target_attainment_percent"]])

run_id,cases_produced,downtime_minutes,operating_minutes,operating_throughput_cph,target_rate_cases_per_hour,target_attainment_percent
R103,290,3.6,56.4,308.4,420,73.4


## 7. Comparing machines or products
**Why it isn't reliable:** each machine made different products, so a machine's performance can't be separated from its products. There are also only 5 runs, all from one morning.

**What would settle it:** more runs, with products run on both machines.

In [8]:
pd.crosstab(pd.Series([r["equipment_id"] for r in production_runs], name="equipment_id"),
            pd.Series([r["product_code"] for r in production_runs], name="product_code"))

product_code,FLOUR-A,FLOUR-B,FLOUR-C
equipment_id,,,
PKG-01,2,0,1
PKG-02,0,2,0
